# ML-02 — ML Task Framing

Lane: Refresh / Content Opportunity Scoring

This notebook frames the Week-2 ML task: decide what to score, define a proxy target, pick a success metric, and show the unit of analysis using the starter CSV. Keep it simple; no model training.

## 1) My lane as an ML task

Task type: SCORING / RANKING.

What is being scored: each page (content item) receives a priority score indicating how urgently it should be reviewed/refreshed.

Decision supported: content/editor team chooses top-K pages to review this sprint.

Action: Editor inspects the top-ranked pages and either Refresh, Review, or Monitor; baseline action label is `Refresh` for top-priority items.

In [ ]:

# Load a small starter dataframe (safe, local)
import pandas as pd
from pathlib import Path
DATA = Path('data/raw/content_refresh_anonymized.csv')
df = pd.read_csv(DATA)
print('Loaded starter CSV, rows =', len(df))
print('One row = one pseudonymized content item')
print('
Columns (sample):')
print(df.columns.tolist()[:20])
print('
Dataframe shape:', df.shape)
df_sample = df[['content_id','client_id','impressions_last_30d','impressions_prev_30d','ctr','avg_position','content_age_days','freshness_tier','trend_direction']].head(10)
df_sample


## 2) Target or proxy

TARGET: Ideally, predict pages that will materially decline (or fail to meet traffic/engagement thresholds) in a future evaluation window — e.g., drop in impressions or sessions over next 30 days after the decision point. This requires a time-forward label (future-window) built from the warehouse.

PROXY (used here as a sketch): `is_declining_proxy` = (trend_direction == 'down') available in the starter snapshot. NOTE: this is a prototype sketch, derived from the same snapshot and should be replaced by a true future-window target when using the full warehouse. It is useful for framing and baseline work only.

In [ ]:

# Sketch the proxy target column in the dataframe
import numpy as np
# reuse df from earlier cell
if 'trend_direction' in df.columns:
    df['is_declining_proxy'] = df['trend_direction'] == 'down'
else:
    df['is_declining_proxy'] = False
# show distribution
print('Proxy target counts:')
print(df['is_declining_proxy'].value_counts(dropna=False))

df[['content_id','impressions_last_30d','impressions_prev_30d','trend_direction','is_declining_proxy']].head(10)


## 3) Success metric

Primary metric: Precision@K (example K=100). Rationale: editorial teams can review a limited top-K each cycle; Precision@K measures how many of those prioritized pages actually meet the review criterion (proxy or true future decline). This matches operational constraints and focuses on top-ranked utility.

## 4) Unit of analysis — real dataframe

ONE ROW = ONE PAGE / CONTENT ITEM. Below is the concrete dataframe sample and a reminder of the intended features that are knowable at decision time: impressions_last_30d, impressions_prev_30d, ctr, avg_position, content_age_days, freshness_tier, word_count, sessions_last_30d. These are observable signals available before any refresh action.

In [ ]:

# show shape, columns, sample rows again for clarity
print('Shape:', df.shape)
print('
Selected columns:')
cols = ['content_id','client_id','impressions_last_30d','impressions_prev_30d','ctr','avg_position','content_age_days','freshness_tier','word_count','sessions_last_30d']
for c in cols:
    if c not in df.columns:
        print('Note: column missing in starter CSV:', c)

print('
Sample rows (10):')
print(df[cols].head(10).to_string(index=False))


## 5) Target column sketch and leakage note

We created `is_declining_proxy` as a sketch above. This uses `trend_direction` present in the starter snapshot — treat as a proxy only.

Eventual true target (full modeling phase): build a future-window label such as: features computed from prior 90 days -> label = drop in impressions (or sessions) larger than X% over the next 30 days (strictly use rows earlier than the label window to avoid leakage). Validate per-client history coverage and seal the final month as test.

## 6) Why ML could beat a fixed rule (short)

A fixed rule (e.g., rank by decline_pct alone) is simple and useful, but limited: it treats all features with fixed weights and cannot easily learn per-client patterns, volume-dependent thresholds, or interactions (e.g., a 50% decline on a 5-impression page is noise; ML can learn to downweight low-volume cases). ML/scoring could potentially combine signals (decline, freshness, position, word_count, query-diversity) and learn non-linear interactions and per-client heterogeneity. If relationships are weak, however, a simple rule may remain competitive; ML is offered as a potential improvement, not a guaranteed win.

## Self-check

- [ ] Task type clearly named
- [ ] Target/proxy clearly defined and labelled
- [ ] Success metric named (Precision@K)
- [ ] Action supported by output (editor review / Refresh)
- [ ] Unit of analysis clearly stated
- [ ] Real dataframe displayed (sample)
- [ ] Target/proxy column sketched
- [ ] No future-window leakage in this framing notebook
- [ ] Explanation: ML vs fixed rule included
- [ ] No model training performed here